In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from jiwer import cer

ocr = pd.read_csv("results/ocr_results.csv")
vlm = pd.read_csv("results/vlm_results.csv")
df  = ocr.merge(vlm, on="id")

# Compute CER where ground truth exists
mask = df['ground_truth'].notna() & (df['ground_truth'] != '')
df.loc[mask, 'cer_raw']  = df[mask].apply(lambda r: cer(r['ground_truth'], r['raw_ocr_text']),  axis=1)
df.loc[mask, 'cer_proc'] = df[mask].apply(lambda r: cer(r['ground_truth'], r['proc_ocr_text']), axis=1)
df.loc[mask, 'cer_vlm']  = df[mask].apply(lambda r: cer(r['ground_truth'], r['vlm_text']),      axis=1)
df.to_csv("results/comparison_table.csv", index=False)

# Plot 1 — Accuracy comparison
methods = ['OCR raw', 'OCR + preprocessing', 'Qwen2-VL 2B']
scores  = [df['cer_raw'].mean(), df['cer_proc'].mean(), df['cer_vlm'].mean()]
colors  = ['#B5D4F4', '#85B7EB', '#378ADD']
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, [1-s for s in scores], color=colors, edgecolor='white', width=0.5)
ax.set_ylabel("Accuracy (1 - CER)")
ax.set_title("OCR vs VLM — table text extraction accuracy")
ax.set_ylim(0, 1)
for bar, s in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{(1-s):.2f}", ha='center', fontsize=10)
plt.tight_layout()
plt.savefig("results/figures/accuracy_by_method.png", dpi=150)
plt.show()

# Plot 2 — Runtime comparison
times = [df['raw_time'].mean(), df['proc_time'].mean(), df['vlm_time'].mean()]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(methods, times, color=['#C0DD97','#97C459','#639922'], edgecolor='white', width=0.5)
ax.set_ylabel("Avg time per image (seconds)")
ax.set_title("Runtime comparison")
plt.tight_layout()
plt.savefig("results/figures/runtime_comparison.png", dpi=150)
plt.show()